# 模块八：DataHub 生产接入 — Phase 2 Kafka 事件流

> **目标**：理解 Phase 1 手工模式的痛点，以及 Phase 2 Kafka 事件流驱动的自动化同步架构。
> **重点**：架构理解，不在实际跑通 Kafka（Demo 环境无 Kafka 集群）。

## 1. 痛点故事：Phase 1 手工模式的问题

Phase 1 使用 `scripts/direct_es_bulk.py` 直接 Bulk API 写入 OpenSearch，绕过 GMS，存在四个核心问题：

1. **新表需人工注册**：每次新增 Parquet 文件，需要手动跑 `emit_browsepaths.py`
2. **元数据更新滞后**：源数据更新后，DataHub UI 不自动刷新
3. **ES 和 GMS 不一致**：`direct_es_bulk.py` 绕过 GMS，OpenSearch 和 MySQL 数据短暂不一致
4. **无法规模化**：5 个系统 × N 张表，手工模式无法支撑长期运营

## 2. 架构解析：Phase 1 vs Phase 2

```
Phase 1（手工模式）：

  scripts/direct_es_bulk.py
         │
         │ 直接 Bulk API
         ▼
   OpenSearch
   (绕过 GMS)
         │
         │ 短暂不一致
         ▼
   GMS MySQL

Phase 2（Kafka 事件流）：

  源数据变更
       │
       ▼
  Kafka Topic
  (MetadataChangeLog_Versioned_v1)
       │
       ▼
  datahub-actions
  (Kafka Consumer)
       │
       ├──► GMS REST API → MySQL（唯一写入路径）
       │
       └──► OpenSearch（通过 GMS 异步同步）
```

## 3. Phase 1 手工模式演示

In [ ]:
# Phase 1: 直接 Bulk API 写入 OpenSearch（绕过 GMS）
import subprocess

result = subprocess.run(
    ["uv", "run", "python", "scripts/direct_es_bulk.py", "--dry-run"],
    capture_output=True, text=True, timeout=60
)
print(result.stdout[-1000:] if len(result.stdout) > 1000 else result.stdout)

## 4. datahub-actions 配置（Phase 2 核心）

datahub-actions 消费 `MetadataChangeLog_Versioned_v1`（非废弃的 `MetadataChangeLog_v4`），
通过 `metadata_change_sync` action 同步到目标 GMS：

```yaml
# scripts/datahub-actions.yml
version: 1
name: "dg-demo-gms-sync-pipeline"
source:
  type: "kafka"
  config:
    topic_routes:
      MetadataChangeLog_Versioned_v1: "mcl"
    connection:
      bootstrap: "localhost:29092"
      schema_registry_url: "http://localhost:28080/schema-registry/api/"
filters:
  - type: "event_type"
    config:
      filter:
        MetadataChangeLogEvent_v1: {}
action:
  type: "metadata_change_sync"
  config:
    gms_server: "http://localhost:28080"
    aspects_to_exclude:
      - "dataHubAccessTokenInfo"
```

> ⚠️ 注意：`datahub-quickstart.yml` 暴露 GMS 端口为 `28080`（映射到容器内 8080），Kafka 为 `29092`，OpenSearch 为 `29200`

## 5. Delta-Lake Ingestion Recipe

## 5. Delta-Lake Ingestion Recipe

配置 `delta-lake` source + `datahub-rest` sink，新增 Parquet 文件自动被发现：

```yaml
# scripts/delta-lake-ingestion.yaml
source:
  type: delta-lake
  config:
    env: "PROD"
    platform_instance: "dg-demo-lakehouse"
    base_path: "/home/szs/Playground/dg-demo/data/lakehouse"
    platform: "delta-lake"
sink:
  type: "datahub-rest"
  config:
    server: "http://localhost:28080"
```

## 6. GMS REST API 直连模式（开发/测试 Fallback）

In [ ]:
# GMS REST API 直连示例（开发/测试 Fallback）
import requests

GMS = "http://localhost:28080"

# 健康检查
r = requests.get(f"{GMS}/health", timeout=5)
print(f"GMS Health: {r.status_code}")

# 读取 dataset（OpenAPI GET，支持）
# 注意：GMS 不支持直接 POST /entities 写入，推荐使用 datahub ingest 或 datahub actions
r = requests.get(f"{GMS}/openapi/entities/v1/latest", params={"urns": "urn:li:dataset:(urn:li:dataPlatform:delta-lake,data/lakehouse/dwd/sap_erp/kna1,PROD)"}, timeout=10)
print(f"GET /openapi/entities/v1/latest: {r.status_code}")
print(f"Aspects: {list(r.json().get('responses',{}).values())[0].get('aspects',{}).keys() if r.status_code==200 else 'N/A'}")

print("\n注意：直接写入请使用 datahub ingest（delta-lake-ingestion.yaml）")
print("或 datahub actions（datahub-actions.yml）")

## 7. 诚实声明：Demo 环境限制

⚠️ **Demo 环境无真实 Kafka 集群**，Phase 2 为架构演示：

- `datahub-actions` 配置参考官方文档（docs.datahub.com/docs/actions）
- 实际运行需要：`datahub docker quickstart` 启动完整栈（GMS + MySQL + Kafka + OpenSearch + datahub-actions）
- Kafka topic 正确名称为 `MetadataChangeLog_Versioned_v1`（非 Module8.md 中所述废弃的 `MetadataChangeLog_v4`）
- Action 类型为 `metadata_change_sync`（非旧版文档中的 `rest`）

## 8. 快速命令汇总

```bash
# 启动完整 DataHub 栈
datahub docker quickstart

# 运行 datahub-actions
datahub actions -c scripts/datahub-actions.yml

# 运行 Delta-Lake ingestion
uv run datahub ingest -c scripts/delta-lake-ingestion.yaml

# GMS 健康检查（端口 28080，映射到容器内 8080）
curl http://localhost:28080/health
```